In [1]:
NO_RECORDS = 102 # more than 100 to test chunking

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
import os
from fastapi import FastAPI, Query, HTTPException
import logging

from lxml import etree as ET
# API calls
import requests
from lxml import etree


from utils.enrichment_methods import get_doi, clean_doi, batch_enrich_subjects_from_openaire, merge_subjects_from_new_method, connect_ids_with_subjects_by_doi


from utils.enrichment_methods import batch_enrich_subjects_from_openalex, batch_enrich_subjects_from_datacite

In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

if (host == '') | (host == "127.0.0.1"):
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        ARRAY(
            SELECT elem->>'subject'
            FROM jsonb_array_elements(datacite_json->'subjects') AS elem
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    doi,
    subjects
FROM query
WHERE 
array_length(subjects, 1) IS NULL AND
-- for these tests, we also want the records to have valid DOIs
     doi IS NOT NULL
ORDER BY id
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results[0])

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_dois = [row["doi"] for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch records")
    raise HTTPException(status_code=500, detail=str(e))

{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481', 'title': 'Water structure around surfactant micelles', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/102734481</identifier>\n    <datestamp>2019-03-18T15:13:33.990Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB620144-1</identifier>\n 

# 1. Check through bigger repositories via DOI

## OpenAIRE

In [7]:
openaire_enriched = batch_enrich_subjects_from_openaire(record_dois)
openaire_enriched[:5]

DEBUG:    2026-07-08 13:29:56 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-08 13:29:56 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.5286%2Fisis.e.rb620144-1%2C10.5286%2Fisis.e.rb1790051-7%2C10.5286%2Fisis.e.rb1830641-1%2C10.5286%2Fisis.e.rb1830641-2%2C10.5286%2Fisis.e.rb1790346-2%2C10.5286%2Fisis.e.rb1790346-3%2C10.5286%2Fisis.e.rb1790346-5%2C10.5286%2Fisis.e.rb1790346-4%2C10.5286%2Fisis.e.rb1790346-6%2C10.5286%2Fisis.e.rb1790346-7 HTTP/1.1" 200 None
DEBUG:    2026-07-08 13:29:57 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-08 13:29:57 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.5286%2Fisis.e.rb1670547-3%2C10.5286%2Fisis.e.rb1990292-1%2C10.5286%2Fisis.e.rb1890354-2%2C10.5286%2Fisis.e.rb1510265-2%2C10.5286%2Fisis.e.rb1920617-3%2C10.5286%2Fisis.e.rb1790175-3%2C10.5286%2Fisis.e.rb1

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': '10.5286/isis.e.rb1790051-7',
  'enriched_subjects': {'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1790346-2',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}}]

In [8]:
print(len(record_dois))

102


In [9]:
succesfully_enriched = [el for el in openaire_enriched if el['enriched_subjects']]
print(len(succesfully_enriched))

24


In [10]:
print(len(succesfully_enriched) / len(record_dois) * 100)

23.52941176470588


In [11]:
initial_set = set([clean_doi(doi) for doi in record_dois])
enriched_set = set([clean_doi(el['doi']) for el in openaire_enriched])

In [12]:
initial_set - enriched_set

set()

In [13]:
enriched_set - initial_set

set()

If both sets are empty, no DOIs have been added during enrichment and all initial DOIs have been preserved even with no subjects.

In [14]:
# To create dictionaries for later joining of subjects:
id_doi_pairs = [{"id": id, "doi": doi}
    for (id, doi) 
    in zip(record_ids[:5], record_dois[:5])]

id_doi_pairs

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218557',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218657',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218664',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224154',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2'}]

In [15]:
connect_ids_with_subjects_by_doi(id_doi_pairs, openaire_enriched)

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218557',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7',
  'enriched_subjects': {'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218657',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218664',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224154',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
  

## OpenAlex API

In [16]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [17]:
test_dois = [
    '10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.34894/YGHDXB',
    '10.15151/esrf-es-1580558984',
    '10.15151/esrf-es-2323561778',
    '10.15151/ESRF-ES-1030397492'
]

In [18]:
batch_enrich_subjects_from_openalex(test_dois, OpenAlex_API_key)

DEBUG:    2026-07-08 13:30:06 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-08 13:30:07 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.1016/j.paid.2021.110888%7Chttps://doi.org/10.23662/fors-ds-686-1%7Chttps://doi.org/10.34894/yghdxb%7Chttps://doi.org/10.15151/esrf-es-1580558984%7Chttps://doi.org/10.15151/esrf-es-2323561778%7Chttps://doi.org/10.15151/esrf-es-1030397492&per_page=100&api_key=nFYjNvBt180q2JQXDpT62V HTTP/1.1" 200 None


[{'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Anxiety',
   'Big Five personality traits',
   'Conscientiousness',
   'Context (archaeology)',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Personality',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'doi': '10.23662/fors-ds-686-1', 'enriched_subjects': set()},
 {'doi': '10.34894/YGHDXB', 'enriched_subjects': set()},
 {'doi': '10.15151/esrf-es-1580558984', 'enriched_subjects': set()},
 {'doi': '10.15151/esrf-es-2323561778', 'enriched_subjects': set()},
 {'doi': '10.15151/ESRF-ES-1030397492', 'enriched_subjects': set()}]

In [19]:
OA_0_0 = batch_enrich_subjects_from_openalex(record_dois,
                                    OpenAlex_API_key,
                                    score_treshold = 0.5)
OA_0_0

DEBUG:    2026-07-08 13:30:07 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-08 13:30:08 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.5286/isis.e.rb620144-1%7Chttps://doi.org/10.5286/isis.e.rb1790051-7%7Chttps://doi.org/10.5286/isis.e.rb1830641-1%7Chttps://doi.org/10.5286/isis.e.rb1830641-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-3%7Chttps://doi.org/10.5286/isis.e.rb1790346-5%7Chttps://doi.org/10.5286/isis.e.rb1790346-4%7Chttps://doi.org/10.5286/isis.e.rb1790346-6%7Chttps://doi.org/10.5286/isis.e.rb1790346-7%7Chttps://doi.org/10.5286/isis.e.rb1670547-3%7Chttps://doi.org/10.5286/isis.e.rb1990292-1%7Chttps://doi.org/10.5286/isis.e.rb1890354-2%7Chttps://doi.org/10.5286/isis.e.rb1510265-2%7Chttps://doi.org/10.5286/isis.e.rb1920617-3%7Chttps://doi.org/10.5286/isis.e.rb1790175-3%7Chttps://doi.org/10.5286/isis.e.rb1990395-3%7Chttps://doi.o

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-5',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-4',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-6',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1670547-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1990292-1',


In [20]:
len([el for el in OA_0_0 if el['enriched_subjects']])

0

In [21]:
OA_0_5 = batch_enrich_subjects_from_openalex(record_dois,
                                    OpenAlex_API_key,
                                    score_treshold = 0.5)
OA_0_5

DEBUG:    2026-07-08 13:30:09 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-08 13:30:09 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.5286/isis.e.rb620144-1%7Chttps://doi.org/10.5286/isis.e.rb1790051-7%7Chttps://doi.org/10.5286/isis.e.rb1830641-1%7Chttps://doi.org/10.5286/isis.e.rb1830641-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-3%7Chttps://doi.org/10.5286/isis.e.rb1790346-5%7Chttps://doi.org/10.5286/isis.e.rb1790346-4%7Chttps://doi.org/10.5286/isis.e.rb1790346-6%7Chttps://doi.org/10.5286/isis.e.rb1790346-7%7Chttps://doi.org/10.5286/isis.e.rb1670547-3%7Chttps://doi.org/10.5286/isis.e.rb1990292-1%7Chttps://doi.org/10.5286/isis.e.rb1890354-2%7Chttps://doi.org/10.5286/isis.e.rb1510265-2%7Chttps://doi.org/10.5286/isis.e.rb1920617-3%7Chttps://doi.org/10.5286/isis.e.rb1790175-3%7Chttps://doi.org/10.5286/isis.e.rb1990395-3%7Chttps://doi.o

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-5',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-4',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-6',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1670547-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1990292-1',


In [22]:
len([el for el in OA_0_5 if el['enriched_subjects']])

0

In [23]:
OA_0_7 = batch_enrich_subjects_from_openalex(record_dois,
                                    OpenAlex_API_key,
                                    score_treshold = 0.7)

DEBUG:    2026-07-08 13:30:10 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-08 13:30:11 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.5286/isis.e.rb620144-1%7Chttps://doi.org/10.5286/isis.e.rb1790051-7%7Chttps://doi.org/10.5286/isis.e.rb1830641-1%7Chttps://doi.org/10.5286/isis.e.rb1830641-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-3%7Chttps://doi.org/10.5286/isis.e.rb1790346-5%7Chttps://doi.org/10.5286/isis.e.rb1790346-4%7Chttps://doi.org/10.5286/isis.e.rb1790346-6%7Chttps://doi.org/10.5286/isis.e.rb1790346-7%7Chttps://doi.org/10.5286/isis.e.rb1670547-3%7Chttps://doi.org/10.5286/isis.e.rb1990292-1%7Chttps://doi.org/10.5286/isis.e.rb1890354-2%7Chttps://doi.org/10.5286/isis.e.rb1510265-2%7Chttps://doi.org/10.5286/isis.e.rb1920617-3%7Chttps://doi.org/10.5286/isis.e.rb1790175-3%7Chttps://doi.org/10.5286/isis.e.rb1990395-3%7Chttps://doi.o

In [24]:
len([el for el in OA_0_7 if el['enriched_subjects']])

0

In [25]:
succesfully_enriched = [el for el in OA_0_7 if el['enriched_subjects']]
print(len(succesfully_enriched)/len(record_dois) * 100)

0.0


## DataCite API

In [26]:
datacite_enriched = batch_enrich_subjects_from_datacite(record_dois)
datacite_enriched[:5]

DEBUG:    2026-07-08 13:30:12 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-08 13:30:12 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.5286/isis.e.rb620144-1?disable-facets=true HTTP/1.1" 200 None
DEBUG:    2026-07-08 13:30:12 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-08 13:30:13 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.5286/isis.e.rb1790051-7?disable-facets=true HTTP/1.1" 200 None
DEBUG:    2026-07-08 13:30:13 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-08 13:30:13 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.5286/isis.e.rb1830641-1?disable-facets=true HTTP/1.1" 200 None
DEBUG:    2026-07-08 13:30:13 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-08 13:30:14 urllib3.connectionpool@544

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': '10.5286/isis.e.rb1790051-7',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1790346-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [27]:
succesfully_enriched = [el for el in datacite_enriched if el['enriched_subjects']]
print(len(succesfully_enriched)/len(record_dois) * 100)

24.509803921568626


## 2. Enrich through Related DOIs  
1) Searching thru our dataset DB for the subjects of the related DOI(s)
2) Sending related DOI(s) thru the external APIs and gathering those subjects

In [28]:
# For a specific DOI list, search our datasetdb and retrieve subjects if any.
# The idea - maybe a dataset is part of another dataset that HAS relevant subjects that can be re-used

# we'll have LIKE in the WHERE part to match a doi that optionally begins with https: etc

query_subjects_by_doi = """
WITH query AS (
    SELECT
    LOWER(doi) AS doi,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects
    FROM public.records
)
SELECT subjects
FROM query
WHERE LOWER(query.doi) LIKE ANY (%s)
LIMIT 100;
"""

In [29]:
def sql_doi_list(doi_list):

    # the starting % will match https://doi.org/<doi> as well as plain <doi>
    # having no % at the end will ensure the <doi> is matched exactly, not just a substring: 10.123 will not match 10.1234
    return [f"%{clean_doi(doi)}" for doi in doi_list]

In [30]:
# Our test dois will serve as the imaginary "related DOIs" of a dataset:

sql_doi_list(test_dois)

['%10.1016/j.paid.2021.110888',
 '%10.23662/fors-ds-686-1',
 '%10.34894/yghdxb',
 '%10.15151/esrf-es-1580558984',
 '%10.15151/esrf-es-2323561778',
 '%10.15151/esrf-es-1030397492']

In [31]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        doi_par = sql_doi_list(test_dois)

        cur = conn.cursor()
        
        cur.execute(query_subjects_by_doi, (doi_par,))
        
        related_subjects = cur.fetchall()
except Exception as e:
    print(f"Query error: {e}")
        

related_subjects

[{'subjects': []},
 {'subjects': ['Arts and Humanities', 'Earth and Environmental Sciences']},
 {'subjects': []},
 {'subjects': []},
 {'subjects': []}]

Let's get the related DOIs from certain records we have in the db db

In [32]:
def get_related_dois(xml_string):
    root = etree.fromstring(xml_string.encode("utf-8"))
    results = root.xpath(
        "//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI']/text()"
    )

    # A single record can have multiple related DOIs
    return [r.strip() for r in results]

In [33]:
# not a lot of records have related dois, I've noted these ones that do:
record_dois_with_related = ['10.34894/8RQMNC',
        '10.34894/9OUUT4',
        '10.34894/ALNHZR',
        '10.34894/CIEDXN',
        '10.34894/EELVV7',
        '10.34894/JE9XVU',
        '10.34894/VJATFE',
        '10.34894/XESS9X']

In [34]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        doi_par = sql_doi_list(record_dois_with_related)

        query = """
            WITH query AS (
                SELECT
                raw_metadata,
                doi
                FROM public.records
            )
            SELECT raw_metadata
            FROM query
            WHERE LOWER(query.doi) LIKE ANY (%s)
            LIMIT 100;
        """

        cur = conn.cursor()
        
        cur.execute(query, (doi_par,))
        
        results = cur.fetchall()
        rec_rel_metadata = [rec["raw_metadata"] for rec in results]
except Exception as e:
    print(f"Query error: {e}")
        
if rec_rel_metadata:
    rec_rel_metadata[0]
else:
    print("None of those DOIs found in database.")

In [35]:
# extract their relations from xml metadata
related_dois = [get_related_dois(metadata) for metadata in rec_rel_metadata]
related_dois

[['10.1016/j.soilbio.2015.11.010'],
 ['10.1016/j.baae.2013.07.002'],
 ['10.1890/14-0563.1'],
 ['10.1111/1462-2920.13235'],
 ['10.1038/ismej.2015.30'],
 ['10.1016/j.ejop.2015.05.003'],
 ['10.1111/1462-2920.12949'],
 ['10.1111/oik.01828']]

In [36]:
# Step 1: search related DOIS thru database
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        related_subjects = []
        # Loop for each record's related list
        for rec_dois in related_dois:
            doi_par = sql_doi_list(rec_dois)

            cur = conn.cursor()
            
            cur.execute(query_subjects_by_doi, (doi_par,))
            
            res = set(cur.fetchall())
            res = {"DatasetDB: " + sub for sub in res}
            related_subjects.append(res)
except Exception as e:
    print(f"Query error: {e}")
        

related_subjects

[set(), set(), set(), set(), set(), set(), set(), set()]

In [37]:
# Step 2 - for each related DOI set try the three external APIs


for ix, rec_dois in enumerate(related_dois):
    oaire = batch_enrich_subjects_from_openaire(rec_dois)

    oaire = set().union(*(item["enriched_subjects"] for item in oaire))
    oaire = {"OpenAIRE: " + sub for sub in oaire} 

    oalex = batch_enrich_subjects_from_openalex(rec_dois, OpenAlex_API_key)
    oalex = set().union(*(item["enriched_subjects"] for item in oalex))
    oalex = {"OpenAlex: " + sub for sub in oalex} 

    datacite = batch_enrich_subjects_from_datacite(rec_dois)
    datacite = set().union(*(item["enriched_subjects"] for item in datacite))
    datacite = {"DataCite: " + sub for sub in datacite} 

    related_subjects[ix] = related_subjects[ix] | oaire | oalex | datacite


related_subjects

DEBUG:    2026-07-08 13:31:13 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-08 13:31:14 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.1016%2Fj.soilbio.2015.11.010 HTTP/1.1" 200 None
DEBUG:    2026-07-08 13:31:14 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-08 13:31:15 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.1016/j.soilbio.2015.11.010&per_page=100&api_key=nFYjNvBt180q2JQXDpT62V HTTP/1.1" 200 None
DEBUG:    2026-07-08 13:31:15 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-08 13:31:16 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.1016/j.soilbio.2015.11.010?disable-facets=true HTTP/1.1" 404 None
DEBUG:    2026-07-08 13:31:16 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openai

[{'OpenAIRE: 03 medical and health sciences',
  'OpenAIRE: 0301 basic medicine',
  'OpenAIRE: 0303 health sciences',
  'OpenAIRE: 106022 Microbiology',
  'OpenAIRE: 106022 Mikrobiologie',
  'OpenAIRE: 15. Life on land',
  'OpenAIRE: AMEBAS ACANTHAMOEBA-CASTELLANII',
  'OpenAIRE: ARBUSCULAR MYCORRHIZAL FUNGI',
  'OpenAIRE: BACTERIAL COMMUNITY COMPOSITION',
  'OpenAIRE: CILIATES PROTOZOA',
  'OpenAIRE: CILIOPHORA',
  'OpenAIRE: Fungi',
  'OpenAIRE: Microcosm studies',
  'OpenAIRE: PLANT-GROWTH',
  'OpenAIRE: REVISED CLASSIFICATION',
  'OpenAIRE: RHIZOSPHERE',
  'OpenAIRE: SDG 15 - Life on Land',
  'OpenAIRE: SDG 15 – Leben an Land',
  'OpenAIRE: SEQUENCE',
  'OpenAIRE: Soil food web',
  'OpenAIRE: Soil metatranscriptomes',
  'OpenAIRE: Soil protists',
  'OpenAIRE: TESTATE AMEBAS',
  'OpenAIRE: Trophic interactions',
  'OpenAIRE: international',
  'OpenAlex: Food web',
  'OpenAlex: Microbial Community Ecology and Physiology',
  'OpenAlex: Plant Pathogens and Fungal Diseases',
  'OpenAlex:

# 3. Extract keywords using statistical approach

Try a YAKE and RAKE library.  These require no training or dictionaries, support multiple languages, domains and text sizes.
Both will default to english stop words - check if that's good enough (test on Croatian titles and descriptions).  

https://github.com/INESCTEC/yake  
https://csurfer.github.io/rake-nltk/_build/html/advanced.html


In [38]:
# Get the records' titles and descriptions. Also get existing subjects to compare with the model extracted ones!

query_records_text = """
WITH query AS (
    SELECT
        records.*,
        ARRAY(
            SELECT elem->>'subject'
            FROM jsonb_array_elements(datacite_json->'subjects') AS elem
        ) AS subjects,
        (
            -- if there are multiple descriptions, concatenate them one after another
            SELECT string_agg(elem->>'description', ' ')
            FROM jsonb_array_elements(datacite_json->'descriptions') AS elem
        ) AS description
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    doi,
    subjects,
    description
FROM query
-- no restriction on number of subjects and DOI availability
ORDER BY id
LIMIT %s
;
"""

In [39]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records_text, (NO_RECORDS,))
        
        results = cur.fetchall()

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_descriptions = [str(row["description"]) for row in  results]
        record_subjects = [row["subjects"] for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

In [40]:
[str(title + ": ---" + desc) for title,desc in zip(record_titles, record_descriptions)][:5]

['DEKISO C17H18Cl4: ---None',
 'Empty CROSTAT no centre stick: ---None',
 '(La,Pb)MnO3 RT w=-159: ---None',
 'Characterisation of Lauryl lactose micelles: ---None',
 'Unidentified Phase Transition in Polar Lanthanum Tantalate (LaTaO4): ---The family of crystals with formula ABX4 represent a very interesting class of ferroelectrics that are extremely two-dimensional. Fluorides have been extensively researched but there is great potential for oxides of this class, such as LaTaO4, due to oxides being favoured for device applications. Studies have reported the evolution of morphologies of LaTaO4 with temperature, but one phase transition at ~200ºC is accompanied by an anomalously small peak in permittivity, which suggests the transition is unusual. Variable temperature XRD of this material has revealed a change in lattice parameters which coincide with the phase transition but this is suspected to be a consequence of a shift in oxygen positions. It is then suggested that by using HRPD, the

In [41]:
record_subjects[:5]

[['1989',
  'Bristol',
  'C17H18Cl4',
  'ccw',
  'DEKISO',
  'JAKH',
  'JAKHoward',
  'RAL',
  'SXD',
  'w=90'],
 ['1623',
  '1990',
  'centre',
  'CROSTAT',
  'Dubna',
  'Empty',
  'JINR',
  'Savenko',
  'stick',
  'SXD'],
 ['(La', '1997', 'ISIS', 'Keen', 'Pb)MnO3', 'SXD', 'w=-159'],
 [],
 ['Physics', 'Chemistry', 'Materials']]

## YAKE

After initial tests - evaluate only records that have a title AND a description.

In [42]:
import yake

In [43]:
kw_extractor = yake.KeywordExtractor()

keywords = []
for title, desc in zip(record_titles, record_descriptions):
    kws = kw_extractor.extract_keywords(title + " " + desc)
    kw_list = []
    for kw, score in kws:
        kw_list.append(str(round(score, 3)) + " - " + kw)
    keywords.append(kw_list)
keywords

[['0.03 - DEKISO'],
 ['0.023 - Empty CROSTAT',
  '0.045 - CROSTAT no centre',
  '0.097 - centre stick',
  '0.145 - CROSTAT',
  '0.158 - Empty',
  '0.297 - centre',
  '0.297 - stick'],
 ['0.045 - La,Pb'],
 ['0.015 - Lauryl lactose micelles',
  '0.023 - Characterisation of Lauryl',
  '0.045 - Lauryl lactose',
  '0.097 - lactose micelles',
  '0.145 - Lauryl',
  '0.158 - Characterisation',
  '0.297 - lactose',
  '0.297 - micelles'],
 ['0.0 - Polar Lanthanum Tantalate',
  '0.005 - Lanthanum Tantalate',
  '0.005 - Polar Lanthanum',
  '0.017 - Unidentified Phase Transition',
  '0.026 - Phase Transition',
  '0.028 - crystals with formula',
  '0.028 - extremely two-dimensional',
  '0.034 - family of crystals',
  '0.043 - Unidentified Phase',
  '0.054 - interesting class',
  '0.062 - Tantalate',
  '0.069 - Transition',
  '0.069 - Phase',
  '0.072 - Polar',
  '0.072 - Lanthanum',
  '0.092 - Transition in Polar',
  '0.117 - class of ferroelectrics',
  '0.126 - oxygen positions',
  '0.14 - Variable